## Task 2: Build Time Series Forecasting Models

### 0. Load Processed Data

We reload the cleaned dataset saved at the end of Task 1, rather than
re-running the full cleaning pipeline. We'll focus on TSLA for
forecasting, since it's the asset we have a "view" on (per Task 4's
approach), while BND and SPY use historical averages.

In [7]:
try:
    combined_df = pd.read_csv('../data/processed/combined_assets.csv', parse_dates=['Date'])
    tsla_df = combined_df[combined_df['Ticker'] == 'TSLA'].sort_values('Date').reset_index(drop=True)
    print(tsla_df.shape)
    print(tsla_df.head())
except Exception as e:
    print(f"Error loading processed data: {e}")

(2888, 8)
        Date  Adj Close      Close       High        Low       Open    Volume  \
0 2015-01-02  14.620667  14.620667  14.883333  14.217333  14.858000  71466000   
1 2015-01-05  14.006000  14.006000  14.433333  13.810667  14.303333  80527500   
2 2015-01-06  14.085333  14.085333  14.280000  13.614000  14.004000  93928500   
3 2015-01-07  14.063333  14.063333  14.318667  13.985333  14.223333  44526000   
4 2015-01-08  14.041333  14.041333  14.253333  14.000667  14.187333  51637500   

  Ticker  
0   TSLA  
1   TSLA  
2   TSLA  
3   TSLA  
4   TSLA  


### 1. Imports

We import `statsmodels`/`pmdarima` for ARIMA/SARIMA, `tensorflow.keras`
for LSTM, and `sklearn` for scaling and evaluation metrics.

In [8]:
try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from statsmodels.tsa.arima.model import ARIMA
    from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
    import pmdarima as pm
    from sklearn.preprocessing import MinMaxScaler
    from sklearn.metrics import mean_absolute_error, mean_squared_error
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense
except Exception as e:
    print(f"Error importing modeling libraries: {e}")

### 2. Prepare Data for Modeling: Chronological Split

We split the data chronologically — **train on 2015–2024, test on
2025–2026** — rather than randomly shuffling. Random splits would leak
future information into training and produce misleadingly good
results, since time series data has temporal dependency (each day
depends on prior days).

In [9]:
try:
    train_df = tsla_df[tsla_df['Date'] < '2025-01-01'].reset_index(drop=True)
    test_df = tsla_df[tsla_df['Date'] >= '2025-01-01'].reset_index(drop=True)

    print(f"Train range: {train_df['Date'].min()} to {train_df['Date'].max()} ({len(train_df)} rows)")
    print(f"Test range: {test_df['Date'].min()} to {test_df['Date'].max()} ({len(test_df)} rows)")

    train_close = train_df['Adj Close']
    test_close = test_df['Adj Close']
except Exception as e:
    print(f"Error splitting train/test data: {e}")

Train range: 2015-01-02 00:00:00 to 2024-12-31 00:00:00 (2516 rows)
Test range: 2025-01-02 00:00:00 to 2026-06-29 00:00:00 (372 rows)


### 3. ACF/PACF Plots

Autocorrelation (ACF) and Partial Autocorrelation (PACF) plots help
visually estimate the `p` (AR order) and `q` (MA order) parameters for
ARIMA before relying on automated search. We plot these on the
differenced series, since prices are non-stationary (confirmed by the
ADF test in Task 1).

In [ ]:
try:
    diff_series = train_close.diff().dropna()

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    plot_acf(diff_series, ax=axes[0], lags=40)
    plot_pacf(diff_series, ax=axes[1], lags=40)
    axes[0].set_title('ACF - Differenced TSLA Close')
    axes[1].set_title('PACF - Differenced TSLA Close')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Error plotting ACF/PACF: {e}")